In [1]:
import pandas as pd
import math
import numpy as np

ROW_COUNT = 3730870


df = pd.read_csv("./datasets/1-09-1-20.csv", nrows=math.floor(0.05 * ROW_COUNT))

df = df.drop(columns=["Unnamed: 0"])  # this is noise
print(df.iloc[0, 2:10])

cols = list(df.columns)
time_data_cols = cols[:2]
new_order = (
    cols[2:] + time_data_cols
)  # put the time columns last so it doesn't get in the way of the column changes below
df = df[new_order]


ask_cols = [
    f"ask_{t}_{i}" for i in range(1, len(cols) // 4 + 1) for t in ["price", "vol"]
]

bid_cols = [
    f"bid_{t}_{i}" for i in range(1, len(cols) // 4 + 1) for t in ["price", "vol"]
]

full_cols = bid_cols + ask_cols


df.columns = full_cols + ["timestamp_ms", "datetime_str"]
df.index = pd.to_datetime(df["timestamp_ms"], unit="ms")
df = df.drop(columns=["timestamp_ms", "datetime_str"])

df[full_cols] = df[full_cols].astype("float32")
print(full_cols)

# print(df.head())

# df.info(memory_usage="deep")

2    17181.6
3     23.371
4    17181.5
5      0.746
6    17181.4
7      5.428
8    17181.2
9       0.89
Name: 0, dtype: object
['bid_price_1', 'bid_vol_1', 'bid_price_2', 'bid_vol_2', 'bid_price_3', 'bid_vol_3', 'bid_price_4', 'bid_vol_4', 'bid_price_5', 'bid_vol_5', 'bid_price_6', 'bid_vol_6', 'bid_price_7', 'bid_vol_7', 'bid_price_8', 'bid_vol_8', 'bid_price_9', 'bid_vol_9', 'bid_price_10', 'bid_vol_10', 'ask_price_1', 'ask_vol_1', 'ask_price_2', 'ask_vol_2', 'ask_price_3', 'ask_vol_3', 'ask_price_4', 'ask_vol_4', 'ask_price_5', 'ask_vol_5', 'ask_price_6', 'ask_vol_6', 'ask_price_7', 'ask_vol_7', 'ask_price_8', 'ask_vol_8', 'ask_price_9', 'ask_vol_9', 'ask_price_10', 'ask_vol_10']


In [2]:
df["micro_price"] = (
    df["bid_vol_1"] * df["ask_price_1"] + df["ask_vol_1"] * df["bid_price_1"]
) / (df["bid_vol_1"] + df["ask_vol_1"])

In [3]:
# ----- e_b logic -----

df["bid_price_1_prev"] = df["bid_price_1"].shift(1)
df["bid_vol_1_prev"] = df["bid_vol_1"].shift(1)

condlist_eb = [
    df["bid_price_1"] > df["bid_price_1_prev"],
    df["bid_price_1"] == df["bid_price_1_prev"],
    df["bid_price_1"] < df["bid_price_1_prev"],
]

choicelist_eb = [
    df["bid_vol_1"],
    df["bid_vol_1"] - df["bid_vol_1_prev"],
    -df["bid_vol_1_prev"],
]

df["e_b"] = np.select(condlist_eb, choicelist_eb, default=0.0)

# ----- e_a logic -----

df["ask_price_1_prev"] = df["ask_price_1"].shift(1)
df["ask_vol_1_prev"] = df["ask_vol_1"].shift(1)

condlist_ea = [
    df["ask_price_1"] > df["ask_price_1_prev"],
    df["ask_price_1"] == df["ask_price_1_prev"],
    df["ask_price_1"] < df["ask_price_1_prev"],
]

choicelist_ea = [
    df["ask_vol_1"],
    df["ask_vol_1"] - df["ask_vol_1_prev"],
    -df["ask_vol_1_prev"],
]

df["e_a"] = np.select(condlist_ea, choicelist_ea, default=0.0)

# ----- OFI logic -----


df["OFI"] = df["e_b"] - df["e_a"]

In [ ]:
k = 40
df["future_mean_price"] = df["micro_price"].rolling(window=k).mean().shift(-k)

df["percentage_return"] = (df["future_mean_price"] - df["micro_price"]) / df[
    "micro_price"
]

epsilon = 5e-5

condlist = [
    df["percentage_return"] > epsilon,
    df["percentage_return"].between(-epsilon, epsilon),
    df["percentage_return"] < -epsilon,
]
choicelist = [1, 0, -1]

df["target"] = np.select(condlist, choicelist, 0)

In [ ]:
df = df.drop(
    columns=[
        "ask_vol_1_prev",
        "bid_vol_1_prev",
        "ask_price_1_prev",
        "bid_price_1_prev",
        "percentage_return",
        "future_mean_price",
    ]
)

In [6]:
print(df.columns)
print(len(df.columns))

Index(['bid_price_1', 'bid_vol_1', 'bid_price_2', 'bid_vol_2', 'bid_price_3',
       'bid_vol_3', 'bid_price_4', 'bid_vol_4', 'bid_price_5', 'bid_vol_5',
       'bid_price_6', 'bid_vol_6', 'bid_price_7', 'bid_vol_7', 'bid_price_8',
       'bid_vol_8', 'bid_price_9', 'bid_vol_9', 'bid_price_10', 'bid_vol_10',
       'ask_price_1', 'ask_vol_1', 'ask_price_2', 'ask_vol_2', 'ask_price_3',
       'ask_vol_3', 'ask_price_4', 'ask_vol_4', 'ask_price_5', 'ask_vol_5',
       'ask_price_6', 'ask_vol_6', 'ask_price_7', 'ask_vol_7', 'ask_price_8',
       'ask_vol_8', 'ask_price_9', 'ask_vol_9', 'ask_price_10', 'ask_vol_10',
       'micro_price', 'e_b', 'e_a', 'OFI', 'target'],
      dtype='object')
45
